In [3]:
%pip install rank_bm25 --no-cache
%pip install sentence-transformers --no-cache
%pip install google-generativeai --no-cache

In [4]:
corpus = [
    "The Transformer architecture uses self-attention mechanisms to weigh the importance of different words in a sequence.",
    "Self-attention allows models to process all words in parallel, unlike RNNs which process them sequentially.",
    "Transformers encode meaning by mapping input tokens into high-dimensional vector spaces using multi-head attention.",
    "Optimization techniques like Stochastic Gradient Descent (SGD) help neural networks minimize loss functions.",
    "AdamW is a popular optimization algorithm that improves upon Adam by decoupling weight decay from the gradient update.",
    "During training, weight decay acts as a regularization technique to prevent overfitting in deep neural networks.",
    "The Softmax function converts a vector of raw scores (logits) into a probability distribution over classes.",
    "In classification tasks, the cross-entropy loss measures the performance of a model whose output is a probability value.",
    "Backpropagation is the fundamental algorithm used to calculate gradients for weight updates in neural networks.",
    "Large Language Models (LLMs) are pre-trained on massive datasets using self-supervised learning objectives."
]

In [5]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        # BM25 Setup
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)
        # SBERT Setup
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.corpus_embeddings = self.encoder.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # 1. BM25 Ranking
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_rank_indices = np.argsort(bm25_scores)[::-1]

        # 2. SBERT Ranking
        query_embedding = self.encoder.encode(query, convert_to_tensor=True)
        sbert_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0]
        sbert_rank_indices = np.argsort(sbert_scores.cpu().numpy())[::-1]

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores = np.zeros(len(self.corpus))
        for rank, idx in enumerate(bm25_rank_indices):
            rrf_scores[idx] += 1 / (self.k + rank + 1)
        for rank, idx in enumerate(sbert_rank_indices):
            rrf_scores[idx] += 1 / (self.k + rank + 1)

        final_ranking = np.argsort(rrf_scores)[::-1][:top_k]

        results = []
        for idx in final_ranking:
            results.append({
                "doc_id": idx,
                "rrf_score": rrf_scores[idx],
                "bm25_rank": np.where(bm25_rank_indices == idx)[0][0] + 1,
                "sbert_rank": np.where(sbert_rank_indices == idx)[0][0] + 1,
                "text": self.corpus[idx]
            })
        return results

In [6]:
from sentence_transformers import CrossEncoder
import google.generativeai as genai

# Setup
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
model = genai.GenerativeModel('gemini-1.5-flash')

def get_hyde_query(query: str) -> str:
    prompt = f"Please write a technical paragraph answering the question: '{query}'"
    response = model.generate_content(prompt)
    return response.text

def rerank(query, candidates, top_k=3):
    pairs = [[query, c['text']] for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, score in enumerate(scores):
        candidates[i]['cross_score'] = score

    reranked = sorted(candidates, key=lambda x: x['cross_score'], reverse=True)
    return reranked[:top_k]

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [7]:
def advanced_rag(user_query: str) -> str:
    # 1. Query Expansion (HyDE)
    hyde_query = get_hyde_query(user_query)

    # 2. Hybrid Retrieval
    retriever = HybridRetriever(corpus)
    initial_candidates = retriever.retrieve(hyde_query, top_k=10)

    # 3. Re-Ranking (using original query)
    final_docs = rerank(user_query, initial_candidates, top_k=3)

    # 4. LLM Generation
    context = "\n".join([d['text'] for d in final_docs])
    prompt = f"Using the context: {context}\nAnswer the query: {user_query}"
    response = model.generate_content(prompt)

    return response.text